In [1]:
# ── 환경 감지 + 경로 설정 ──────────────────────────────────────
import os

IS_KAGGLE = os.path.exists('/kaggle')

if IS_KAGGLE:
    PREP_DIR    = '/kaggle/input/notebooks/hongsh184/test122/preprocessed/'
    MODELS_DIR  = '/kaggle/input/notebooks/hongsh184/test1222/models/'
    MODELS_DIR2 = '/kaggle/input/notebooks/hongsh184/test122222/models/'
    OUTPUT_DIR  = '/kaggle/working/'
    DATA_DIR    = '/kaggle/input/datasets/hongsh184/deepguard-cicids2017-2/'
    DDOS_DIR    = '/kaggle/input/datasets/hongsh184/afternoon-ddos/'
    SAVE_MODELS_DIR       = OUTPUT_DIR + 'models/'
    SAVE_PREPROCESSED_DIR = OUTPUT_DIR + 'preprocessed/'
else:
    BASE        = 'D:/Deep_Learning/DeepGuard/'
    PREP_DIR    = BASE + 'preprocessed/'
    MODELS_DIR  = BASE + 'models/'
    MODELS_DIR2 = BASE + 'models/'
    OUTPUT_DIR  = BASE + 'visualizations/'
    DATA_DIR    = BASE + 'data/CICIDS2017/'
    DDOS_DIR    = BASE + 'data/CICIDS2017/'
    SAVE_MODELS_DIR       = BASE + 'models/'
    SAVE_PREPROCESSED_DIR = BASE + 'preprocessed/'

print(f"환경: {'Kaggle' if IS_KAGGLE else 'Local'}")
print(f"PREP_DIR: {PREP_DIR}")
print(f"MODELS_DIR: {MODELS_DIR}")


환경: Local
PREP_DIR: D:/Deep_Learning/DeepGuard/preprocessed/
MODELS_DIR: D:/Deep_Learning/DeepGuard/models/


In [1]:
# ── Section 1. 라이브러리 임포트 ──────────────────────────────
import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split
import pickle
import os

print("라이브러리 임포트 완료")


라이브러리 임포트 완료


In [2]:
# ── Section 2. 데이터 로드 ────────────────────────────────────
monday_df    = pd.read_csv(DATA_DIR + 'Monday-WorkingHours.pcap_ISCX.csv')
tuesday_df   = pd.read_csv(DATA_DIR + 'Tuesday-WorkingHours.pcap_ISCX.csv')
wednesday_df = pd.read_csv(DATA_DIR + 'Wednesday-workingHours.pcap_ISCX.csv')
friday_df    = pd.read_csv(DATA_DIR + 'Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv')
print("Monday shape   :", monday_df.shape)
print("Tuesday shape  :", tuesday_df.shape)
print("Wednesday shape:", wednesday_df.shape)
print("Friday shape   :", friday_df.shape)


Monday shape   : (529918, 79)
Tuesday shape  : (445909, 79)
Wednesday shape: (692703, 79)
Friday shape   : (225745, 79)


In [3]:
# ── Section 3. 컬럼명 공백 제거 ───────────────────────────────
for df in [monday_df, tuesday_df, wednesday_df, friday_df]:
    df.columns = df.columns.str.strip()

print("\nTuesday Label:\n",   tuesday_df['Label'].value_counts())
print("\nWednesday Label:\n", wednesday_df['Label'].value_counts())
print("\nFriday Label:\n",    friday_df['Label'].value_counts())



Tuesday Label:
 Label
BENIGN         432074
FTP-Patator      7938
SSH-Patator      5897
Name: count, dtype: int64

Wednesday Label:
 Label
BENIGN              440031
DoS Hulk            231073
DoS GoldenEye        10293
DoS slowloris         5796
DoS Slowhttptest      5499
Heartbleed              11
Name: count, dtype: int64

Friday Label:
 Label
DDoS      128027
BENIGN     97718
Name: count, dtype: int64


In [4]:
# ── Section 4. 클렌징 ─────────────────────────────────────────
drop_cols = ['Label', 'Flow ID', 'Source IP', 'Destination IP', 'Timestamp']

def clean(df, is_test=False):
    labels = None
    if is_test:
        labels = df['Label'].copy()
    df = df.drop(columns=[c for c in drop_cols if c in df.columns])
    df.replace([np.inf, -np.inf], np.nan, inplace=True)
    df.dropna(inplace=True)
    if is_test:
        labels = labels.loc[df.index].values
    return df, labels

monday_clean, _               = clean(monday_df,    is_test=False)
tuesday_clean, tuesday_labels = clean(tuesday_df,   is_test=True)
wednesday_clean, wed_labels   = clean(wednesday_df, is_test=True)
friday_clean, fri_labels      = clean(friday_df,    is_test=True)

print("Monday 클렌징 후  :", monday_clean.shape)
print("Tuesday 클렌징 후 :", tuesday_clean.shape)
print("Wednesday 클렌징 후:", wednesday_clean.shape)
print("Friday 클렌징 후  :", friday_clean.shape)


Monday 클렌징 후  : (529481, 78)
Tuesday 클렌징 후 : (445645, 78)
Wednesday 클렌징 후: (691406, 78)
Friday 클렌징 후  : (225711, 78)


In [5]:
# ── Section 5. 스케일링 ───────────────────────────────────────
scaler = MinMaxScaler()
scaler.fit(monday_clean)  # 정상 데이터로만 fit

monday_scaled    = scaler.transform(monday_clean)
tuesday_scaled   = scaler.transform(tuesday_clean)
wednesday_scaled = scaler.transform(wednesday_clean)
friday_scaled    = scaler.transform(friday_clean)

os.makedirs(SAVE_MODELS_DIR, exist_ok=True)
with open(SAVE_MODELS_DIR + 'scaler.pkl', 'wb') as f:
    pickle.dump(scaler, f)

print("스케일링 완료")


스케일링 완료


In [6]:
# ── Section 6a. Sliding Window W5 ────────────────────────────
def create_sequences(data, window_size):
    return np.array([data[i:i+window_size] for i in range(len(data) - window_size)])

W1 = 5
X_tr_w5     = create_sequences(monday_scaled,    W1)
X_te_tue_w5 = create_sequences(tuesday_scaled,   W1)
X_te_wed_w5 = create_sequences(wednesday_scaled, W1)
X_te_fri_w5 = create_sequences(friday_scaled,    W1)
y_te_tue_w5 = tuesday_labels[W1:]
y_te_wed_w5 = wed_labels[W1:]
y_te_fri_w5 = fri_labels[W1:]

X_train_w5, X_val_w5 = train_test_split(X_tr_w5, test_size=0.1, random_state=42)
X_train_w5 = X_train_w5[:len(X_train_w5)//2]
X_val_w5   = X_val_w5[:len(X_val_w5)//2]

os.makedirs(SAVE_PREPROCESSED_DIR, exist_ok=True)
np.save(SAVE_PREPROCESSED_DIR + 'X_train_w5.npy',  X_train_w5)
np.save(SAVE_PREPROCESSED_DIR + 'X_val_w5.npy',    X_val_w5)
np.save(SAVE_PREPROCESSED_DIR + 'X_te_tue_w5.npy', X_te_tue_w5)
np.save(SAVE_PREPROCESSED_DIR + 'X_te_wed_w5.npy', X_te_wed_w5)
np.save(SAVE_PREPROCESSED_DIR + 'X_te_fri_w5.npy', X_te_fri_w5)
np.save(SAVE_PREPROCESSED_DIR + 'y_te_tue_w5.npy', y_te_tue_w5)
np.save(SAVE_PREPROCESSED_DIR + 'y_te_wed_w5.npy', y_te_wed_w5)
np.save(SAVE_PREPROCESSED_DIR + 'y_te_fri_w5.npy', y_te_fri_w5)

# 메모리 해제
del X_tr_w5, X_train_w5, X_val_w5
del X_te_tue_w5, X_te_wed_w5, X_te_fri_w5
print("W5 저장 완료!")


W5 저장 완료!


In [7]:
# ── Section 6b. Sliding Window W20 ───────────────────────────
def create_sequences(data, window_size):
    return np.array([data[i:i+window_size] for i in range(len(data) - window_size)])

W2 = 20
X_tr_w20     = create_sequences(monday_scaled,    W2)

# 테스트 데이터 샘플링 (메모리 절약)
X_te_tue_w20 = create_sequences(tuesday_scaled[:50000],   W2)
X_te_wed_w20 = create_sequences(wednesday_scaled[:50000], W2)
X_te_fri_w20 = create_sequences(friday_scaled[:50000],    W2)
y_te_tue_w20 = tuesday_labels[W2:50000]
y_te_wed_w20 = wed_labels[W2:50000]
y_te_fri_w20 = fri_labels[W2:50000]

X_train_w20, X_val_w20 = train_test_split(X_tr_w20, test_size=0.1, random_state=42)
X_train_w20 = X_train_w20[:len(X_train_w20)//2]
X_val_w20   = X_val_w20[:len(X_val_w20)//2]

del X_tr_w20

np.save(SAVE_PREPROCESSED_DIR + 'X_train_w20.npy',  X_train_w20)
np.save(SAVE_PREPROCESSED_DIR + 'X_val_w20.npy',    X_val_w20)
np.save(SAVE_PREPROCESSED_DIR + 'X_te_tue_w20.npy', X_te_tue_w20)
np.save(SAVE_PREPROCESSED_DIR + 'X_te_wed_w20.npy', X_te_wed_w20)
np.save(SAVE_PREPROCESSED_DIR + 'X_te_fri_w20.npy', X_te_fri_w20)
np.save(SAVE_PREPROCESSED_DIR + 'y_te_tue_w20.npy', y_te_tue_w20)
np.save(SAVE_PREPROCESSED_DIR + 'y_te_wed_w20.npy', y_te_wed_w20)
np.save(SAVE_PREPROCESSED_DIR + 'y_te_fri_w20.npy', y_te_fri_w20)

print("W20 저장 완료!", os.listdir(SAVE_PREPROCESSED_DIR))


W20 저장 완료! ['X_te_fri_w5.npy', 'y_te_fri_w20.npy', 'X_te_tue_w5.npy', 'y_te_fri_w5.npy', 'X_val_w20.npy', 'X_te_fri_w20.npy', 'X_te_wed_w20.npy', 'X_val_w5.npy', 'X_train_w20.npy', 'X_train_w5.npy', 'y_te_wed_w5.npy', 'y_te_wed_w20.npy', 'X_te_tue_w20.npy', 'X_te_wed_w5.npy', 'y_te_tue_w20.npy', 'y_te_tue_w5.npy']
